# Kolmogorov–Smirnov Tests: One and Two Independent Samples

**STA 507 1.0 Nonparametric Statistics** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta507.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [The empirical distribution](#theory-01)
- [One-sample hypotheses and requirements](#theory-02)
- [Constructing the statistic](#theory-03)
- [Decisions and estimated parameters](#theory-04)
- [Ordinal, rounded and grouped observations](#theory-05)
- [Two independent samples](#theory-06)
- [Direction, meaning and follow-up](#theory-07)

## Detailed theory · One- and two-sample Kolmogorov–Smirnov tests

<a id="theory-01"></a>

### The empirical distribution
The empirical CDF is $F_n(x)=n^{-1}\sum I(X_i\le x)$. It is a nondecreasing step function with jumps at observations. A CDF is a cumulative probability, not a density or a histogram height. K–S detects differences in centre, spread or shape, rather than a particular mean difference.

<a id="theory-02"></a>

### One-sample hypotheses and requirements
Test $H_0:F(x)=F_0(x)$ for all x against a discrepancy somewhere. The classical reference assumes independent observations from a continuous distribution and a fully specified $F_0$. Specifying a normal family but estimating its mean and SD from the same sample does not leave the ordinary K–S null distribution unchanged.

<a id="theory-03"></a>

### Constructing the statistic
Sort $x_{(1)}\le\cdots\le x_{(n)}$. The empirical CDF just before a unique observation is $(i-1)/n$ and just after it is $i/n$:
$$D^+=\max_i\{i/n-F_0(x_{(i)})\},\quad
D^-=\max_i\{F_0(x_{(i)})-(i-1)/n\},\quad D=\max(D^+,D^-).$$
Checking only the after-jump absolute difference can miss the maximum. The worked table includes both sides.

<a id="theory-04"></a>

### Decisions and estimated parameters
Large D is evidence against fit. Use finite-sample quantiles or a p-value matching n and the alternative. Large-sample cutoffs proportional to $1/\sqrt n$ are approximations. If parameters were fitted, use a calibration designed for that situation, such as an appropriate fitted-normal test or parametric bootstrap that refits each simulated sample.

<a id="theory-05"></a>

### Ordinal, rounded and grouped observations
The lecture applications include ordered data. With discrete values, repeated rounding or ordinal categories, the classical continuous reference is not generally exact. A justified discrete-null calculation, simulation or permutation method is needed. With grouped counts the individual within-bin values are unknown; do not create an exact-looking individual K–S test by inventing them.

<a id="theory-06"></a>

### Two independent samples
For empirical CDFs $F_m$ and $G_n$, test equality of the two population distributions using
$$D_{m,n}=\sup_x|F_m(x)-G_n(x)|.$$
Pool unique observed values, calculate both cumulative fractions at each jump and take their maximum gap. Groups must be independent; the data are not paired. With continuous distributions, exact or suitable asymptotic references are available. Ties require attention to calibration.

<a id="theory-07"></a>

### Direction, meaning and follow-up
A one-sided CDF alternative has a direction that is opposite to intuitive value ordering: a population with smaller measurements often has a larger CDF at a given x. Define the CDF inequality explicitly. A two-sided significant result establishes a distributional discrepancy, not which single parameter caused it. Plot both CDFs and report where the largest gap occurs.

## Concepts covered

**Source alignment:** Handout 1: single-sample Kolmogorov–Smirnov goodness-of-fit (week 3). Handout 3: Single Sample Kolmogorov–Smirnov Goodness of Fit Test.

**Mission:** Match each design and measurement scale to a test, then explain the result in context.

## Worked calculation · One-sample K–S test: empirical CDF working table

### Question and assumptions

Test five independent continuous measurements against the fully specified Uniform(0,1) distribution. The null parameters are not estimated from these data.

### Procedure

1. Sort the observations.
2. Calculate F0(xi), i/n and (i−1)/n for each sorted value.
3. Calculate D+=max(i/n−F0(xi)) and D−=max(F0(xi)−(i−1)/n). Take D=max(D+,D−).
4. Compare with the correct finite-sample critical value or p-value for this null.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| i | x(i) | F0(x) | i/n | (i−1)/n | D+ candidate | D− candidate |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 0.08 | 0.08 | 0.2 | 0 | 0.12 | 0.08 |
| 2 | 0.21 | 0.21 | 0.4 | 0.2 | 0.19 | 0.01 |
| 3 | 0.37 | 0.37 | 0.6 | 0.4 | 0.23 | -0.03 |
| 4 | 0.64 | 0.64 | 0.8 | 0.6 | 0.16 | 0.04 |
| 5 | 0.92 | 0.92 | 1 | 0.8 | 0.08 | 0.12 |

| Statistic | Value |
| --- | --- |
| D+ | 0.23 |
| D− | 0.12 |
| D | 0.23 |
| Finite-n 5% critical D (n=5) | 0.563275 |
| Decision at .05 | Do not reject specified null |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


x=[.08,.21,.37,.64,.92]; n=len(x)
rows=[[i,z,z,i/n,(i-1)/n,i/n-z,z-(i-1)/n] for i,z in enumerate(sorted(x),1)]
show_table(["i","x(i)","F0(x)","i/n","(i−1)/n","D+ candidate","D− candidate"],rows)
Dp=max(row[5] for row in rows); Dm=max(row[6] for row in rows); D=max(Dp,Dm)
critical=.5632751983660635  # Finite-n two-sided 5% cutoff for n=5.
show_table(["Statistic","Value"],[["D+",Dp],["D−",Dm],["D",D],
    ["Finite-n 5% critical D (n=5)",critical],
    ["Decision at .05","Reject specified null" if D>critical else "Do not reject specified null"]])
assert abs(D-.23)<1e-12


### Decision and interpretation

The largest gap may occur just before an empirical CDF jump, which is why both columns are needed. The displayed finite-sample cutoff applies only to this n=5, two-sided 5% continuous-null example. For a different n or alpha, recalculate it using scipy.stats.kstwo.ppf(1-alpha,n). Use scipy.stats.kstest(x, "uniform", method="exact") for an exact-null p-value; fitting null parameters requires a different calibration.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(507)

In [ ]:
x=rng.normal(0,1,60)
result=stats.kstest(x,'norm',args=(0,1))
print('K-S statistic:',round(result.statistic,3),'p-value:',round(result.pvalue,4))
xs=np.sort(x); fn=np.arange(1,len(x)+1)/len(x); f0=stats.norm.cdf(xs)
plt.step(xs,fn,where='post',label='empirical CDF'); plt.plot(xs,f0,label='specified N(0,1) CDF'); plt.legend(); plt.title('Largest vertical CDF gap is D'); plt.show()

### Practice · Distribution fit
1. Define Dₙ in words. What does a large value mean?
2. Why must F₀ be specified before testing?
3. A normal distribution is fitted using the same sample, then ordinary one-sample K–S is run. What is the calibration issue?

<details><summary>Answer sketches</summary>
1. The largest absolute vertical difference between empirical and hypothesized CDFs; larger values mean worse fit. 2. The null reference distribution assumes the hypothesized CDF is fixed. 3. Parameter fitting changes the null distribution of D; use an adjusted test or bootstrap.
</details>